# 8장. 지표의 불확실성

이 지표의 값은 자료가 달라지면 얼마나 달라지는가.

**준비:** 6장에서 쓴 경기도 「일반음식점(중국식) 현황」 CSV를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 끝까지 도는 데 삼사 분이 걸린다.

## 8.1 부트스트랩으로 지표의 신뢰구간 구하기

In [ ]:
import glob
import numpy as np
import pandas as pd

pd.set_option("display.unicode.east_asian_width", True)
pd.set_option("display.width", 200)

def first_line(path):
    with open(path, encoding="cp949") as f:
        return f.readline()

name = [f for f in glob.glob("*.csv") if first_line(f).startswith("시군명,사업장명")][0]
raw = pd.read_csv(name, encoding="cp949", dtype=str)

raw["시군"] = raw["시군명"].str.strip()
raw["구분"] = np.where(raw["시군명"].str.endswith(" "), "공백 있음", "공백 없음")
shops = raw.sort_values(["구분", "영업상태명"], ascending=[True, False]).drop_duplicates(["시군", "사업장명", "인허가일자"]).copy()

shops["개업"] = pd.to_datetime(shops["인허가일자"], format="%Y%m%d", errors="coerce")
shops["폐업"] = pd.to_datetime(shops["폐업일자"], format="%Y%m%d", errors="coerce")
shops = shops.dropna(subset=["개업"]).reset_index(drop=True)
END = max(shops["개업"].max(), shops["폐업"].max())
shops["끝"] = shops["폐업"].fillna(END + pd.Timedelta(days=1))
shops["위도"] = pd.to_numeric(shops["WGS84위도"], errors="coerce")
shops["경도"] = pd.to_numeric(shops["WGS84경도"], errors="coerce")
shops["지번"] = shops["소재지지번주소"].str.replace(r"\s+", " ", regex=True).str.extract(r"^(.*?\d+(?:-\d+)?)")[0]
print(len(raw), len(shops), END.date())

In [ ]:
def snapshot(day):
    """day에 영업 중이던 가게마다 한 줄. 특성은 그날 알 수 있는 값으로만 만들고, 그 뒤 3년 안에 폐업했는지를 레이블로 붙인다."""
    day = pd.Timestamp(day)
    began, ended = shops["개업"].to_numpy(), shops["끝"].to_numpy()
    open_now = (began < day.to_datetime64()) & (ended > day.to_datetime64())
    gone = shops["폐업"].notna().to_numpy() & (ended < day.to_datetime64())
    lately = gone & (ended >= (day - pd.DateOffset(years=3)).to_datetime64())

    a = shops[open_now].copy()
    a["기준일"] = day
    a["업력"] = (day - a["개업"]).dt.days / 365.25
    a["지번폐업"] = a["지번"].map(shops[gone].groupby("지번").size()).fillna(0)      # 같은 지번에서 먼저 폐업한 중국집 수

    def within(mask, metres):
        """a의 가게마다, mask에 해당하는 가게 가운데 metres 안에 있는 것의 수."""
        lat, lon = shops.loc[mask, "위도"].to_numpy(), shops.loc[mask, "경도"].to_numpy()
        dy = (a["위도"].to_numpy()[:, None] - lat[None, :]) * 111195
        dx = (a["경도"].to_numpy()[:, None] - lon[None, :]) * 111195 * np.cos(np.radians(a["위도"].to_numpy()))[:, None]
        return np.where(a["위도"].isna(), np.nan, (dx ** 2 + dy ** 2 < metres ** 2).sum(axis=1))

    a["주변"] = within(open_now, 500) - 1            # 반경 500m 안에서 영업 중인 다른 중국집
    a["주변폐업"] = within(lately, 1000)              # 반경 1km 안에서 지난 3년 사이 폐업한 중국집
    a["폐업함"] = (a["폐업"].notna() & (a["폐업"] <= day + pd.DateOffset(years=3))).astype(int)
    return a

train = pd.concat([snapshot(d) for d in ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03"]], ignore_index=True)
test = snapshot("2023-04-03").reset_index(drop=True)
y = test["폐업함"].to_numpy()

print(train.groupby("기준일")["폐업함"].agg(["size", "mean"]).round(3).to_string())
print(len(test), int(y.sum()), round(float(y.mean()), 3))

In [ ]:
SIGUN = sorted(shops["시군"].unique())
YEARS = [0, 1, 2, 3, 5, 10, 20]

def onehot(s, cats):
    return (np.asarray(s)[:, None] == np.asarray(cats)[None, :]).astype(float)

def columns(df):
    """업력 구간 가변수 7개, 시군 가변수 30개, 같은 지번의 폐업 이력, 주변의 영업 중·최근 폐업 가게 수(로그), 좌표 없음 표시."""
    return np.column_stack([
        onehot(np.digitize(df["업력"], YEARS) - 1, range(len(YEARS))),
        onehot(df["시군"], SIGUN)[:, 1:],
        np.minimum(df["지번폐업"], 3),
        np.log1p(df["주변"].fillna(0)), np.log1p(df["주변폐업"].fillna(0)), df["주변"].isna().astype(float),
    ])

def auc(actual, s):
    rank = pd.Series(s).rank().to_numpy()
    n1 = actual.sum()
    return float((rank[actual == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * (len(actual) - n1)))

X, Xt, ytr = columns(train), columns(test), train["폐업함"].to_numpy()
score = Xt @ np.linalg.lstsq(X, ytr, rcond=None)[0]
age = -test["업력"].to_numpy()                     # 업력이 짧을수록 높은 점수를 주는 규칙
print(X.shape, round(auc(y, score), 3), round(auc(y, age), 3))

In [ ]:
rng = np.random.default_rng(0)
n = len(y)

def span(values):
    """95% 백분위수 구간: 값들의 2.5% 지점과 97.5% 지점."""
    return np.quantile(values, [0.025, 0.975]).round(3).tolist()

again = []
for _ in range(2000):
    pick = rng.integers(0, n, n)                   # 6,311곳에서 6,311곳을 복원 추출한다
    again.append(auc(y[pick], score[pick]))
again = np.array(again)
print(round(float(again.mean()), 4), round(float(again.std()), 4), span(again))
print(len(np.unique(pick)), round(len(np.unique(pick)) / n, 3))

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)

In [ ]:
# 그림 8-2
lo, hi = span(again)
fig, ax = plt.subplots(figsize=(7, 3.9))
counts, edges, _ = ax.hist(again, bins=40, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.6)
top = counts.max()
ax.set_ylim(0, top * 1.32)
for v, text, ha in [(lo, f"2.5% 지점\n{lo:.3f}", "right"), (hi, f"97.5% 지점\n{hi:.3f}", "left")]:
    ax.axvline(v, color=BRICK, lw=1.2, ls=(0, (4, 3)))
    ax.text(v + (-0.0008 if ha == "right" else 0.0008), top * 1.17, text, fontsize=8, color=BRICK, ha=ha, va="center", linespacing=1.4)
ax.axvline(auc(y, score), color=TEAL, lw=1.6)
ax.text(auc(y, score), top * 1.17, f"원래 시험 자료\n{auc(y, score):.3f}", fontsize=8, color=TEAL, ha="center", va="center", linespacing=1.4, bbox=BOX)
ax.annotate("", (hi, top * 0.5), (lo, top * 0.5), arrowprops={"arrowstyle": "<->", "color": BRICK, "lw": 1.0})
ax.text((lo + hi) / 2, top * 0.54, "2,000개 가운데 95%가 이 구간에 있다", fontsize=8, color=BRICK, ha="center", bbox=BOX)
ax.set_xlabel("복원 추출한 시험 자료에서 계산한 AUC", fontsize=8)
ax.set_ylabel("횟수(전체 2,000번)", fontsize=8)
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_2.png", dpi=300)
plt.show()

In [ ]:
A, n1, n0 = auc(y, score), int(y.sum()), int((1 - y).sum())
q1, q2 = A / (2 - A), 2 * A ** 2 / (1 + A)
se = np.sqrt((A * (1 - A) + (n1 - 1) * (q1 - A ** 2) + (n0 - 1) * (q2 - A ** 2)) / (n1 * n0))
print(round(float(se), 4), [round(float(A - 1.96 * se), 3), round(float(A + 1.96 * se), 3)])

for reps in [100, 500, 2000]:
    print(reps, [span([auc(y[pick], score[pick]) for pick in (np.random.default_rng(seed).integers(0, n, (reps, n)))]) for seed in (1, 2, 3)])

In [ ]:
def top_tenth(actual, s):
    return float(actual[np.argsort(-s)[: len(actual) // 10]].mean() * 100)

hits = [top_tenth(y[pick], score[pick]) for pick in (rng.integers(0, n, n) for _ in range(2000))]
print(round(top_tenth(y, score), 1), np.quantile(hits, [0.025, 0.975]).round(1).tolist(), round(float(np.std(hits)), 2))

In [ ]:
SIZES = [200, 400, 800, 1600, 3200, 6311]
spread = {}
for size in SIZES:
    few = [auc(y[pick], score[pick]) for pick in (rng.integers(0, n, size) for _ in range(1000))]
    spread[size] = float(np.std(few))
    print(size, span(few), round(spread[size], 4))

In [ ]:
# 그림 8-3
fig, ax = plt.subplots(figsize=(7, 3.8))
sizes = np.array(SIZES)
sd = np.array([spread[s_] for s_ in SIZES])
ax.plot(sizes, sd[-1] * np.sqrt(sizes[-1] / sizes), color=GRAY, lw=1.2, ls=(0, (4, 3)), label="1 ÷ √n 에 비례하는 선(6,311곳의 값에 맞춤)")
ax.plot(sizes, sd, color=NAVY, lw=1.8, marker="o", ms=5, label="부트스트랩으로 구한 AUC의 표준오차")
for s_, v in zip(sizes, sd):
    ax.text(s_, v * 1.13, f"{v:.4f}", ha="center", fontsize=8, color=NAVY)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xticks(sizes, [f"{s_:,}" for s_ in sizes])
ax.set_yticks([0.01, 0.02, 0.05], ["0.01", "0.02", "0.05"])
ax.minorticks_off()
ax.set_ylim(0.007, 0.085)
ax.set_xlabel("시험 자료의 크기 n(가게 수, 로그 눈금)", fontsize=8)
ax.set_ylabel("AUC의 표준오차(로그 눈금)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper right")
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_8_3.png", dpi=300)
plt.show()

## 8.2 두 모델의 차이: 짝지은 부트스트랩

In [ ]:
def paired(s1, s2, reps=2000):
    """같은 복원 추출 표본에서 두 점수의 AUC를 함께 구한다."""
    a, b = [], []
    for _ in range(reps):
        pick = rng.integers(0, n, n)
        a.append(auc(y[pick], s1[pick]))
        b.append(auc(y[pick], s2[pick]))
    return np.array(a), np.array(b)

def compare(name, a, b):
    gap = a - b
    rho = float(np.corrcoef(a, b)[0, 1])
    print(name, span(a), span(b))
    print("  차이", span(gap), "양수인 비율", round(float((gap > 0).mean()), 3))
    print("  표준편차", round(float(a.std()), 4), round(float(b.std()), 4), "상관", round(rho, 3),
          "차이의 표준편차", round(float(gap.std()), 4), round(float(np.sqrt(a.var() + b.var() - 2 * rho * a.std() * b.std())), 4))

a_age, b_age = paired(score, age)
compare("모델, 업력 규칙", a_age, b_age)

In [ ]:
keep = list(range(38))                              # 주변 가게에 관한 세 열(38~40번)을 뺀다
lean = Xt[:, keep] @ np.linalg.lstsq(X[:, keep], ytr, rcond=None)[0]
print(round(auc(y, lean), 3))

a_lean, b_lean = paired(score, lean)
compare("모델, 주변을 뺀 모델", a_lean, b_lean)

In [ ]:
# 그림 8-4
fig, ax = plt.subplots(figsize=(7, 4.0))
bins = np.arange(-0.035, 0.0251, 0.001)
ax.hist(a_age - b_age, bins=bins, color=to_rgba(NAVY, 0.22), edgecolor=NAVY, lw=0.5, label="모델 - 업력 규칙 (원래 차이 -0.005)")
ax.hist(a_lean - b_lean, bins=bins, color=to_rgba(BRICK, 0.5), edgecolor=BRICK, lw=0.5, label="모델 - 주변을 뺀 모델 (원래 차이 +0.005)")
ax.axvline(0, color=TEAL, lw=1.4)
top = ax.get_ylim()[1]
ax.set_ylim(0, top * 1.12)
ax.text(0.0006, top * 1.04, "차이 0", fontsize=8, color=TEAL)
g1, g2 = span(a_age - b_age), span(a_lean - b_lean)
ax.text(-0.034, top * 0.62, f"95% 구간 [{g1[0]:+.3f}, {g1[1]:+.3f}]\n0을 포함한다", fontsize=8, color=NAVY, linespacing=1.5)
ax.text(0.0115, top * 0.62, f"95% 구간 [{g2[0]:+.3f}, {g2[1]:+.3f}]\n0을 포함하지 않는다", fontsize=8, color=BRICK, linespacing=1.5)
ax.set_xlabel("같은 복원 추출 표본에서 구한 두 AUC의 차이", fontsize=8)
ax.set_ylabel("횟수(전체 2,000번)", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_4.png", dpi=300)
plt.show()

In [ ]:
# 그림 8-5
fig, ax = plt.subplots(figsize=(5.8, 5.4))
ax.plot([0.56, 0.64], [0.56, 0.64], color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.scatter(a_age, b_age, s=5, color=to_rgba(NAVY, 0.35), lw=0, label=f"세로축: 업력 규칙 (상관 {np.corrcoef(a_age, b_age)[0, 1]:.2f})")
ax.scatter(a_lean, b_lean, s=5, color=to_rgba(BRICK, 0.55), lw=0, label=f"세로축: 주변을 뺀 모델 (상관 {np.corrcoef(a_lean, b_lean)[0, 1]:.2f})")
ax.text(0.632, 0.6365, "두 AUC가 같은 선", fontsize=8, color="#6b7480", rotation=45, ha="right")
ax.text(0.562, 0.636, "선보다 위: 세로축의 방법이 더 높다", fontsize=8, color="#6b7480")
ax.text(0.638, 0.5625, "선보다 아래: 모델이 더 높다", fontsize=8, color="#6b7480", ha="right")
ax.set_xlim(0.56, 0.64); ax.set_ylim(0.56, 0.64); ax.set_aspect("equal")
ax.set_xlabel("모델의 AUC(복원 추출 표본마다 점 하나)", fontsize=8)
ax.set_ylabel("비교 대상의 AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="lower right", bbox_to_anchor=(1.0, 0.06), markerscale=3)
frame(ax, grid="both")
fig.tight_layout()
fig.savefig("fig_8_5.png", dpi=300)
plt.show()

## 8.3 훈련 자료에 따른 변동과 교차검증

In [ ]:
name = (train["시군"] + "|" + train["사업장명"] + "|" + train["인허가일자"]).to_numpy()
shop_ids, which = np.unique(name, return_inverse=True)
rows_of = [np.where(which == g)[0] for g in range(len(shop_ids))]      # 가게마다 그 가게에 해당하는 줄 번호
print(len(train), len(shop_ids))

retrained, both = [], []
for _ in range(300):
    chosen = rng.integers(0, len(shop_ids), len(shop_ids))            # 가게를 복원 추출하고, 뽑힌 가게의 줄을 모두 가져온다
    rows = np.concatenate([rows_of[g] for g in chosen])
    s = Xt @ np.linalg.lstsq(X[rows], ytr[rows], rcond=None)[0]
    retrained.append(auc(y, s))
    pick = rng.integers(0, n, n)
    both.append(auc(y[pick], s[pick]))
print("훈련 자료만", span(retrained), round(float(np.std(retrained)), 4))
print("훈련 자료와 시험 자료 모두", span(both), round(float(np.std(both)), 4))
print(round(float(np.sqrt(np.var(retrained) + again.var())), 4))

In [ ]:
def crossval(seed, parts=5):
    """가게를 다섯 묶음으로 나눠, 한 묶음을 빼고 나머지로 모델을 추정한 뒤 뺀 묶음에서 AUC를 구한다."""
    fold = np.random.default_rng(seed).integers(0, parts, len(shop_ids))[which]
    marks = []
    for j in range(parts):
        seen, held = fold != j, fold == j
        marks.append(auc(ytr[held], X[held] @ np.linalg.lstsq(X[seen], ytr[seen], rcond=None)[0]))
    return np.array(marks)

one = crossval(0)
print(one.round(3).tolist(), round(float(one.mean()), 3), round(float(one.std(ddof=1)), 3), round(float(one.std(ddof=1) / np.sqrt(5)), 3))
CV = np.array([crossval(s) for s in range(10)])
print(CV.mean(axis=1).round(3).tolist(), round(float(CV.mean()), 3), round(float(CV.mean(axis=1).std(ddof=1)), 4))

In [ ]:
# 그림 8-7
fig, ax = plt.subplots(figsize=(7, 3.9))
for i, marks in enumerate(CV):
    ax.scatter(np.full(5, i), marks, s=22, color=to_rgba(NAVY, 0.3), edgecolor=NAVY, lw=0.6, zorder=2)
    ax.plot([i - 0.28, i + 0.28], [marks.mean()] * 2, color=BRICK, lw=2.2, zorder=3)
ax.axhline(auc(y, score), color=TEAL, lw=1.4)
ax.text(-0.4, auc(y, score) + 0.003, f"2023년 시험 자료의 AUC {auc(y, score):.3f}", fontsize=8, color=TEAL, ha="left")
ax.axhline(CV.mean(), color=BRICK, lw=0.8, ls=(0, (4, 3)), label=f"열 번의 평균 {CV.mean():.3f}")
ax.scatter([], [], s=22, color=to_rgba(NAVY, 0.3), edgecolor=NAVY, lw=0.6, label="묶음 하나의 AUC")
ax.plot([], [], color=BRICK, lw=2.2, label="다섯 묶음의 평균")
ax.set_xticks(range(10), [f"{i}" for i in range(10)])
ax.set_ylim(0.535, 0.625)
ax.set_xlabel("묶음을 나눌 때 쓴 난수의 시작값", fontsize=8)
ax.set_ylabel("AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left", ncol=3)
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_7.png", dpi=300)
plt.show()

## 8.4 기준일에 따른 변동: 시점이 바뀔 때의 지표

In [ ]:
DAYS = ["2011-04-03", "2014-04-03", "2017-04-03", "2020-04-03", "2023-04-03"]
part = {d: train[train["기준일"] == d] for d in DAYS[:-1]}
part[DAYS[-1]] = test

rows, GAPS = [], {}
for j in range(1, 5):
    known, exam = pd.concat([part[d] for d in DAYS[:j]]), part[DAYS[j]]
    s = columns(exam) @ np.linalg.lstsq(columns(known), known["폐업함"].to_numpy(), rcond=None)[0]
    truth, young = exam["폐업함"].to_numpy(), -exam["업력"].to_numpy()
    m = len(truth)
    picks = [rng.integers(0, m, m) for _ in range(1000)]
    gap = [auc(truth[k], s[k]) - auc(truth[k], young[k]) for k in picks]
    GAPS[DAYS[j][:4]] = span(gap)
    rows.append((DAYS[j][:4], len(known), m, round(auc(truth, s), 3), str(span([auc(truth[k], s[k]) for k in picks])),
                 round(auc(truth, young), 3), str(span(gap))))
years = pd.DataFrame(rows, columns=["시험 기준일", "훈련 줄 수", "시험 가게 수", "모델", "모델의 95% 구간", "업력 규칙", "모델 − 업력 규칙"]).set_index("시험 기준일")
print(years.to_string())

In [ ]:
# 그림 8-8
import ast

fig, ax = plt.subplots(figsize=(7, 4.0))
x = np.arange(len(years))
lo = [ast.literal_eval(v)[0] for v in years["모델의 95% 구간"]]
hi = [ast.literal_eval(v)[1] for v in years["모델의 95% 구간"]]
ax.vlines(x - 0.08, lo, hi, color=NAVY, lw=2.4)
ax.plot(x - 0.08, years["모델"], "o", color=NAVY, ms=7, label="모델(세로 막대는 부트스트랩 95% 구간)")
ax.plot(x + 0.08, years["업력 규칙"], "s", color=BRICK, ms=7, label="업력 규칙")
for xi, (a_, b_) in enumerate(zip(years["모델"], years["업력 규칙"])):
    ax.text(xi - 0.16, a_, f"{a_:.3f}", ha="right", va="center", fontsize=8, color=NAVY)
    ax.text(xi + 0.16, b_, f"{b_:.3f}", ha="left", va="center", fontsize=8, color=BRICK)
ax.axhline(0.5, color=GRAY, lw=0.9, ls=(0, (4, 3)))
ax.text(3.45, 0.503, "무작위(0.5)", fontsize=8, color="#6b7480", ha="right")
ax.set_xticks(x, [f"{t}년 기준일" for t in years.index], fontsize=8.5)
ax.set_xlim(-0.5, 3.5); ax.set_ylim(0.49, 0.645)
ax.set_ylabel("AUC", fontsize=8)
ax.legend(frameon=False, fontsize=8, loc="upper left")
frame(ax)
fig.tight_layout()
fig.savefig("fig_8_8.png", dpi=300)
plt.show()